In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
campaigns_bronze = '/Volumes/main/lakehouse_marketing/bronze/campaigns/'


df_campaigns = spark.read\
                    .format("delta")\
                    .option("header", "true")\
                    .load(campaigns_bronze)

display(df_campaigns)
display(df_campaigns.printSchema())

campaign_id,campaign_name,channel,start_date,end_date,ingestion_timestamp,source_file
eb2263dd-87c5-421e-ac24-a3c5c754108f,Campaign_0,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00000-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-420-1-c000.csv
5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Campaign_1,SOCIAL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00000-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-420-1-c000.csv
3da9c2a9-0ed4-4f1a-bd4c-bf374eb93eff,Campaign_2,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00000-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-420-1-c000.csv
d0e6e660-7c69-4ee1-bb5e-4bcf15ed6269,Campaign_3,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00001-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-421-1-c000.csv
a8e56e0c-20de-435d-a031-d750c40db9b4,Campaign_4,social,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00001-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-421-1-c000.csv
9b49bd26-df57-459a-8715-a10343dac043,Campaign_5,social,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00001-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-421-1-c000.csv
b09b2a5c-badc-432a-8159-0f538a0f4efb,Campaign_6,e-mail,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00001-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-421-1-c000.csv
5f987c71-a65e-488e-abf3-ad39fec21bbe,Campaign_7,SOCIAL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00002-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-422-1-c000.csv
1064005c-3985-43cf-bf76-be1d1efa2197,Campaign_8,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00002-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-422-1-c000.csv
01d74256-3860-4ab6-96a4-02f23ae8cc93,Campaign_9,e-mail,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00002-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-422-1-c000.csv


root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- start_date: string (nullable = true)
 |-- end_date: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
df = df_campaigns.select(
    'campaign_id',
    'campaign_name',
    'channel',
    'start_date',
    'end_date',
    'ingestion_timestamp',
    'source_file'
)


df_typed = df\
            .withColumn('campaign_name', F.trim(F.col('campaign_name')))\
            .withColumn('start_date', F.to_date('start_date'))\
            .withColumn('end_date', F.to_date('end_date'))

df_typed.printSchema()

root
 |-- campaign_id: string (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- start_date: date (nullable = true)
 |-- end_date: date (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:
display(df_campaigns.groupBy("channel").count())

channel,count
e-mail,6
EMAIL,10
email,3
SOCIAL,5
social,6


* Normalization

  * `Redefinir a regra para os canais no documento.` 

In [0]:
df_norm = df_typed.withColumn(
    "channel",
    F.when(F.lower(F.regexp_replace("channel", "-", "")) == "email", "EMAIL")
    .when(F.lower(F.col("channel")).isin("social"), "SOCIAL")
    .otherwise("OTHER")
)

display(df_norm.groupBy('channel').count())

channel,count
EMAIL,19
SOCIAL,11


#### Remove `campaign_id` nulos

In [0]:
# Regra 1: campaign_id não pode ser nulo
df_valid = df_norm.filter(F.col("campaign_id").isNotNull())

# Regra 2: start_date <= end_date
df_valid = df_valid.filter(F.col("start_date") <= F.col("end_date"))

df_valid.show()

+--------------------+-------------+-------+----------+----------+--------------------+--------------------+
|         campaign_id|campaign_name|channel|start_date|  end_date| ingestion_timestamp|         source_file|
+--------------------+-------------+-------+----------+----------+--------------------+--------------------+
|eb2263dd-87c5-421...|   Campaign_0|  EMAIL|2025-12-29|2025-12-29|2026-01-13 11:28:...|dbfs:/Volumes/mai...|
|5cec4eb5-edd9-483...|   Campaign_1| SOCIAL|2025-12-29|2025-12-29|2026-01-13 11:28:...|dbfs:/Volumes/mai...|
|3da9c2a9-0ed4-4f1...|   Campaign_2|  EMAIL|2025-12-29|2025-12-29|2026-01-13 11:28:...|dbfs:/Volumes/mai...|
|d0e6e660-7c69-4ee...|   Campaign_3|  EMAIL|2025-12-29|2025-12-29|2026-01-13 11:28:...|dbfs:/Volumes/mai...|
|a8e56e0c-20de-435...|   Campaign_4| SOCIAL|2025-12-29|2025-12-29|2026-01-13 11:28:...|dbfs:/Volumes/mai...|
|9b49bd26-df57-459...|   Campaign_5| SOCIAL|2025-12-29|2025-12-29|2026-01-13 11:28:...|dbfs:/Volumes/mai...|
|b09b2a5c-badc-432.

In [0]:
window = Window.partitionBy("campaign_id").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = (
    df_valid
    .withColumn("row_number", F.row_number().over(window))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print(f"Sem deduplicação: {df_valid.count()}")
print(f"Com deduplicação: {df_dedup.count()}")

Sem deduplicação: 30
Com deduplicação: 30


In [0]:
silver_path = '/Volumes/main/lakehouse_marketing/silver/campaigns'

df_dedup.write\
    .format("delta")\
    .mode("overwrite")\
    .save(silver_path)
    

* Validação

In [0]:
spark.read.format("delta").load(silver_path).display()

campaign_id,campaign_name,channel,start_date,end_date,ingestion_timestamp,source_file
01d74256-3860-4ab6-96a4-02f23ae8cc93,Campaign_9,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00002-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-422-1-c000.csv
03c72ba8-d605-4770-8a63-f881ffd0f9d5,Campaign_19,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00005-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-418-1-c000.csv
080aadfb-e7c9-4b26-9141-25c63a9bedd4,Campaign_10,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00002-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-422-1-c000.csv
0f844fef-1931-49ee-a56c-0941fbf24050,Campaign_15,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00004-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-417-1-c000.csv
1064005c-3985-43cf-bf76-be1d1efa2197,Campaign_8,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00002-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-422-1-c000.csv
118a9d29-2f92-4996-99f1-95d014822f53,Campaign_27,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00007-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-419-1-c000.csv
14fcdd54-9e8f-4965-8a2c-827e98326856,Campaign_29,SOCIAL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00007-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-419-1-c000.csv
1825bc54-30be-445f-a835-14f2ceb81f9d,Campaign_13,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00003-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-416-1-c000.csv
2a25a888-0f02-4ad0-a706-7ef466aa9385,Campaign_21,SOCIAL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00005-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-418-1-c000.csv
310c0c00-3fa7-4104-9bf9-0e27dc96925e,Campaign_16,EMAIL,2025-12-29,2025-12-29,2026-01-13T11:28:26.615Z,dbfs:/Volumes/main/lakehouse_marketing/raw/campaigns/part-00004-tid-1294539358196327925-d0716bbd-f4ac-4599-925c-832725649942-417-1-c000.csv


In [0]:
spark.read.format("delta").load(silver_path).count()

30